In [ ]:
from swmm_api import SwmmReport, SwmmInput
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "serif",
})

## Report Analysis

In [ ]:
nodes = gpd.read_file('nodes.geojson')
subcatchments = gpd.read_file('subcatchments.geojson')

In [ ]:
flood_grid = gpd.read_file("GIS/FLOOD_GRID.geojson")
flood_grid.drop('index_right', axis=1, inplace=True)

In [ ]:
rpt = SwmmReport('60_1.5_30.rpt')

### Hydrologic Risk Computation

In [ ]:
runoff = gpd.GeoDataFrame(rpt.subcatchment_runoff_summary, geometry=subcatchments.set_index('SWMM_ID').geometry)
runoff['runoff_per_ha'] = runoff['Total_Runoff_10^6 ltr'] * 1000 / (runoff.geometry.area / 10_000) # m^3 per ha
runoff['pct_sump'] = subcatchments.set_index('SWMM_ID').pct_sump / 100
runoff['near_floodplain'] = subcatchments.set_index('SWMM_ID').near_floodplain

In [ ]:
a = 0.3
b = 0.1
runoff['hydrologic_risk'] = runoff['Runoff_Coeff'] * (1 + a * runoff['pct_sump'] + b * runoff.near_floodplain) / (1 + a + b)

In [ ]:
joined = gpd.overlay(flood_grid, runoff, how='intersection')
joined['area'] = joined.geometry.area
grouped = joined.groupby('grid_id').apply(lambda df: pd.Series({'hydrologic_risk': (df['area'] * df['hydrologic_risk']).sum() / df['area'].sum()}), include_groups=False)
flood_grid = flood_grid.merge(grouped, left_on='grid_id', right_index=True, how='left')

In [ ]:
data = runoff["Runoff_Coeff"].dropna()
mean_val = data.mean()
median_val = data.median()

fig, ax = plt.subplots(figsize=(6, 4))

ax.hist(
    data,
    bins=40,
    color="0.8",
    edgecolor="0.3",
    linewidth=0.6
)

ax.set_xlabel("Runoff Coefficient (C)")
ax.set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("runoff_coeff_hist_styled.png")
plt.show()


In [ ]:
subcatchments['total_runoff'] = runoff.reset_index()['Total_Runoff_mm']

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

subcatchments.plot(
    column="total_runoff",
    ax=ax,
    cmap="Blues",
    linewidth=0.01,
    edgecolor="0.7",
    legend=True,
    legend_kwds={
        "shrink": 0.7,
        "label": "Runoff Depth (mm)",
        "orientation": "vertical"
    }
)

ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.scatter(
    subcatchments["pct_impervious"],
    subcatchments["total_runoff"],
    s=8,
    alpha=0.05,
    color="0.2",
    edgecolor="none"
)

ax.set_xlabel("Imperviousness (%)")
ax.set_ylabel("Runoff Depth (mm)")
ax.grid(color="0.9", linestyle="--", linewidth=0.6)

plt.tight_layout()
plt.show()


### Hydraulic Risk Computation

In [ ]:
nodes = nodes.merge(
    rpt.node_flooding_summary[['Total_Flood_Volume_10^6 ltr']],
    left_on="SWMM_ID",
    right_index=True,
    how="left"
)

joined = gpd.sjoin(
    nodes.dropna(subset=['Total_Flood_Volume_10^6 ltr']), 
    flood_grid, 
    how="inner", 
    predicate="within"
)

mean_rate = (
    joined.groupby("index_right")['Total_Flood_Volume_10^6 ltr'].sum()
)
flood_grid["hydraulic_risk"] = (
    flood_grid.index.map(mean_rate)
    .fillna(0)
)



### Combined Risk

In [ ]:
flood_grid['flood_risk'] = (0.5 * np.log(1+flood_grid.hydraulic_risk) + flood_grid.hydrologic_risk)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

flood_grid.plot(
    column="flood_risk",
    ax=ax,
    cmap="Reds",
    linewidth=0.2,
    edgecolor="0.7",
    legend=True,
    legend_kwds={
        "shrink": 0.7,
        "label": "Flood Risk ",
        "orientation": "vertical"
    }
)

ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
flood_grid.flood_risk.corr(flood_grid.hydraulic_risk)**2

In [ ]:
flood_grid.hydraulic_risk.corr(flood_grid.hydrologic_risk)**2

In [ ]:
flood_grid.flood_risk.hist(bins=50)

In [ ]:
flood_grid.plot(column='flood_risk', cmap='Reds', legend=True)

In [ ]:
vulnerability = gpd.read_file('GIS/VULNERABILITY.geojson').to_crs(flood_grid.crs)

In [ ]:
attr = "VULNERABILITY_INDEX"

filtered = gpd.sjoin(
    flood_grid,
    subcatchments,
    predicate="intersects",
    how="inner"
)

inter = gpd.overlay(filtered, vulnerability, how="intersection")
inter["_area"] = inter.geometry.area
A = inter.groupby("grid_id")["_area"].sum()                
W = (inter["_area"] * inter[attr]).groupby(inter["grid_id"]).sum()

weighted = (W / A)
weighted.name = attr
flood_grid[attr] = flood_grid["grid_id"].map(weighted)

In [ ]:
flood_grid['most_vulnerable'] = (flood_grid.VULNERABILITY_INDEX * flood_grid.flood_risk)

In [ ]:
flood_grid.VULNERABILITY_INDEX.corr(flood_grid.flood_risk)

In [ ]:
flood_grid.plot(column='most_vulnerable', cmap='Blues', legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
flood_grid.plot(ax=ax, color='none', edgecolor='k', linewidth=0.5)
runoff.plot(ax=ax, column='hydrologic_risk', legend=True)
ax.set_axis_off()